# Validation Notebook for My Candidate ReRank Model
In this notebook, we attempted to develop a two-stage model which includes the candidate generation model (Covisitation Matrix) and Ranking Model. 
This practice is widely used in big tech company since the candidate generation 

It should be noted that the candidate generation model should target for high recall while the ranking model should target for to rank the most relevent item first.


# Introduction of Notebook


## Step 1: Model Training
### Step 1.1 - Loading Training Data
The training data in this notebook is extracted by this logic: <br>
`train_df = train_df[train_df['session']%10 == 1]` <br>

The label of the training data is stored in the test_labels.parquet, which **contain the label for both training and testing data (For quick experimet only).**


### Step 1.2 - Feature Engineering on Training Data
All the feature is pre-calculated and saved in parquet files. All parquet file is saved in this kaggle dataset `/kaggle/input/otto-validation` <br>
Here we only do a simple joining between the training data & pre-calculated features


### Step 1.3 - Model Training on Training data
We train the LGBM ranking using the training data.

<br>

## Step 2: Model Inference
### Step 2.1 - Loading testing data
The training data in this notebook is extracted by this logic: <br>
`test_df = test_df[test_df['session']%10 == 0]`


### Step 2.2 - Generate Candidates
We use the logic from the Chris Deotte's Candidate ReRank Model to generate 40 potential candidates. We have pre-calculated all the co-visitation dictionary and saved in the kaggle dataset 
`/kaggle/input/otto-covisitation-matrix-parquet-files` <br>
In this stage, we will generate 40 candidates per session and pass into the ranker for the final ranking. <br>

Please refer to Chris notebook for detailed logic explaination
https://www.kaggle.com/code/cdeotte/candidate-rerank-model-lb-0-575 <br>

### Step 2.3 - Ranker Model
1. The recommmended aid from the candidate generation phase will merge with the test aid for ranking.
2. Feature preprocessing step is exactly the same with the training pipeline
3. The test dataframe will be passed for prediction and the scores will be merged with the dataframe
4. Sort each session in the dataframe by score from high to low
5. Using `groupby('session').last(20) to extract top 20 result


### Step 2.4 - Export to CSV
<br>

## Step 3: Model Evaluation
Same logic with the Chris notebook: https://www.kaggle.com/code/cdeotte/compute-validation-score-cv-565

# Credits
1. The validation part of the notebook comes from Chris Deotte Notebook: https://www.kaggle.com/code/cdeotte/compute-validation-score-cv-565 <br>
2. The covisition matrix calculation comes from Chris Deotte Notebook: https://www.kaggle.com/code/cdeotte/candidate-rerank-model-lb-0-575 <br>
3. The code of LGBM ranker is inspired by RADEK OSMULSKI in his Notebook: https://www.kaggle.com/code/radek1/polars-proof-of-concept-lgbm-ranker <br>

# Important Note!
This notebook is still in development phase so it is using the local validation dataset instead of the full training and testing dataset

In [16]:
! pip install polars pandas -qU

In [17]:
VER = 6

import pandas as pd, numpy as np
from tqdm.notebook import tqdm
import os, sys, pickle, glob, gc
from collections import Counter
import itertools
from gensim.models import Word2Vec


# 0:clicks 1:carts 2:orders
type_weight = {0:0.5,
               1:9,
               2:0.5}
type_weight_multipliers = type_weight

clicks_th = 15 
carts_th  = 20 
orders_th = 20 

VER = 7

type_labels = {'clicks':0, 'carts':1, 'orders':2}

# Step 1: Model Training

## Step 1.1 - Generate Training Data

There are two types of aid we will reocommned to the users: 
1. Data from the training data set (Actual aid in the dataset [i.e. the actual behavior of the user. User may cart the item that they clicked or they may click again for same item])
2. Data from candiate generation (Recommended aid from the candidate generation logic [i.e. The item that is related to the actual user behavior])

Therefore, our ranker should be able to rank these two items at the same time.

We want to simulate the this data for inference time with recommended candidates. Therefore, we need to generate some recommended item  in the training size to make sure training set has same distribution as inference time. <br>


### Generating Training data from the test set (Actual behavior)

In [ ]:
# Getting the actual aid from dataset

def load_train_data_sampled():    
    dfs = []
    for e, chunk_file in enumerate(glob.glob('../input/otto-validation/test_parquet/*')):
        chunk = pd.read_parquet(chunk_file)
        chunk.ts = (chunk.ts/1000).astype('int32')
        chunk['type'] = chunk['type'].map(type_labels).astype('int8')
        dfs.append(chunk)
        
    train_df = pd.concat(dfs).reset_index(drop=True) #.astype({"ts": "datetime64[ms]"})

    # Using different sample as the candidate generation for training
    train_df = train_df[train_df['session']%10 == 1]
    return train_df

train_df = load_train_data_sampled()
print('Sampled Training data has shape',train_df.shape)


# This indicate that this aid is actual behaviors
train_df['real_action'] = 1

# CG stands for candidate generation. Since the aid here is actual user behavior, they should have no ranking
train_df['CG_ranking'] = 0

# Only focus on click action first
train_df_click = train_df[train_df['type'] == 0]

# Calculate the last three aid for the embedding calculation
train_df_click['aid_last'] = train_df_click.groupby(['session']).aid.shift(1).bfill()
train_df_click['aid_second_last'] = train_df_click.groupby(['session']).aid.shift(2).bfill()
train_df_click['aid_third_last'] = train_df_click.groupby(['session']).aid.shift(3).bfill()

In [ ]:
train_df_click = train_df_click.with_columns([
    pd.col('aid_last').cast(pl.Int32),
    pd.col('aid_second_last').cast(pl.Int32),
    pd.col('aid_third_last').cast(pl.Int32),
])

### Generating Training data from candidate generation

The below block of code is exactly the same code we used for candidate generation based on the candidate ReRank Model. <br>
You may refer here for the detailed logic: https://www.kaggle.com/code/cdeotte/candidate-rerank-model-lb-0-575

In [20]:
%%time
# Generating recommended aid from the actual aid based on candidate generation
top_clicks = train_df.loc[train_df['type']== 0,'aid'].value_counts().index.values[:20] 


# Improved speed for 2X using polars. 
def pqt_to_dict(path):
    return pl.read_parquet(path).groupby('aid_x').agg(pl.col('aid_y').list()).to_pandas().set_index('aid_x').aid_y.apply(list).to_dict()

DISK_PIECES = 4

# LOAD THREE CO-VISITATION MATRICES
top_20_clicks = pqt_to_dict(f'/kaggle/input/otto-covisitation-matrix-parquet-files/top_20_clicks_v{VER}_0.pqt')

for k in range(1,DISK_PIECES): 
    top_20_clicks.update(pd.read_parquet(f'/kaggle/input/otto-covisitation-matrix-parquet-files/top_20_clicks_v{VER}_{k}.pqt') ) 

def suggest_clicks(df):
    # USER HISTORY AIDS AND TYPES
    aids=df.aid.tolist()
    types = df.type.tolist()
    unique_aids = list(dict.fromkeys(aids[::-1] ))
    # RERANK CANDIDATES USING WEIGHTS
    if len(unique_aids)>=20:
        weights=np.logspace(0.1,1,len(aids),base=2, endpoint=True)-1
        aids_temp = Counter() 
        # RERANK BASED ON REPEAT ITEMS AND TYPE OF ITEMS
        for aid,w,t in zip(aids,weights,types): 
            aids_temp[aid] += w * type_weight_multipliers[t]
        sorted_aids = [k for k,v in aids_temp.most_common(20)]
        return sorted_aids
    # USE "CLICKS" CO-VISITATION MATRIX
    aids2 = list(itertools.chain(*[top_20_clicks[aid] for aid in unique_aids if aid in top_20_clicks]))
    # RERANK CANDIDATES
    top_aids2 = [aid2 for aid2, cnt in Counter(aids2).most_common(40) if aid2 not in unique_aids]    
    result = unique_aids + top_aids2#[:20 - len(unique_aids)]
    # USE TOP20 TEST CLICKS
    return result + list(top_clicks)#[:20-len(result)]


pred_df_clicks = train_df.sort_values(["session", "ts"]).groupby(["session"]).apply(
    lambda x: suggest_clicks(x)
)

train_df_click_recommended = pl.from_pandas(pd.DataFrame(pred_df_clicks, columns = ['aid']).reset_index()).explode("aid")
train_df_click_recommended = train_df_click_recommended.with_columns([
    pl.lit(0).alias('ts').cast(pl.Int32),    
    pl.lit(0).alias('type').cast(pl.Int8),
    pl.lit(0).alias('real_action').cast(pl.Int64),
    pl.col('session').cast(pl.Int32),
    pl.col('aid').cast(pl.Int32),
])
n_col_after_join = train_df_click_recommended.groupby('session').agg([
    pl.col('aid').cumcount().alias('CG_ranking')]).select(
    pl.col('CG_ranking').explode().cast(pl.Int64))
train_df_click_recommended = pl.concat([train_df_click_recommended, n_col_after_join], how="horizontal")

NameError: name 'pl' is not defined

## Step 1.2 - Feature Engineering

### Calculating the sparse feature

In [21]:
model = Word2Vec.load("/kaggle/input/ottoprecalculatedfeatureparquet/word2vec.model")
embedding_weight = np.load('/kaggle/input/ottoprecalculatedfeatureparquet/word2vec.model.wv.vectors.npy')
embedding_weight_neg = np.load('/kaggle/input/ottoprecalculatedfeatureparquet/word2vec.model.syn1neg.npy')

embedding_weigh_dict_df = pl.from_pandas(pd.DataFrame(embedding_weight, columns = ['Embedding_' + str(x) for x in range(32)]).reset_index().rename(columns = {'index':'aid'})).with_columns(pl.col('aid').cast(pl.Int32))

NameError: name 'pl' is not defined

In [ ]:
%%time

# Calculating the embedding of last three actions aid
train_df_click = train_df_click.join(embedding_weigh_dict_df, on = 'aid', how = 'left').join(embedding_weigh_dict_df, right_on = 'aid', left_on = 'aid_last', how = 'left', suffix = 'last_1').join(embedding_weigh_dict_df, right_on = 'aid', left_on = 'aid_second_last', how = 'left', suffix = 'last_2').join(embedding_weigh_dict_df, right_on = 'aid', left_on = 'aid_third_last', suffix = 'last_3').select(pl.exclude(['aid_last', 'aid_second_last', 'aid_third_last']))

In [ ]:
# Calculating the embedding of last three actions aid for the recommended action. We all use the last three actual aid embedding for the last three aid
train_df_click_last_action_embedding = train_df_click.sort(['session', 'ts']).groupby(['session']).last().select(pl.exclude(['aid', 'ts','type','real_action', 'CG_ranking', 'aid_last', 'aid_second_last', 'aid_third_last'] + ['Embedding_' + str(i) for i in range(32)]))

train_df_click_recommended = train_df_click_recommended.join(embedding_weigh_dict_df, on = 'aid', how = 'left').join(train_df_click_last_action_embedding, on = 'session', how = 'left')

### Calculating the dense feature

In [ ]:
# Creating feature by joining with pre-computed feature
aid_global_counter_all_types = pl.read_parquet('/kaggle/input/ottoprecalculatedfeatureparquet/aid_global_counter_all_types.pqt')

aid_global_user_counter_all_types = pl.read_parquet('/kaggle/input/ottoprecalculatedfeatureparquet/aid_global_user_counter_all_types.pqt')

aid_global_user_counter_all_types_time_weighted = pl.read_parquet('/kaggle/input/ottoprecalculatedfeatureparquet/aid_global_user_counter_all_types_time_weighted (1).pqt')

In [ ]:
train_df_click = train_df_click.join(aid_global_counter_all_types, on='aid', suffix ='_global_counter').join(aid_global_user_counter_all_types, on='aid', suffix ='_user_counter').join(aid_global_user_counter_all_types_time_weighted, on='aid', suffix ='_timed_global_counter')
train_df_click_recommended = train_df_click_recommended.join(aid_global_counter_all_types, on='aid', suffix ='_global_counter').join(aid_global_user_counter_all_types, on='aid', suffix ='_user_counter').join(aid_global_user_counter_all_types_time_weighted, on='aid', suffix ='_timed_global_counter')

### Merging two training dataset together

In [ ]:
# Merging two type of data as training data
train_df_click_all = pl.concat([train_df_click_recommended, train_df_click], how = 'vertical')

# Step 1.3 - Model Training

In [ ]:
# Merging the Ground Truth label with training dataset
# Using negative downsampling of 50%

train_labels = pd.read_parquet('../input/otto-validation/test_labels.parquet')
train_labels['type'] = train_labels['type'].map(type_labels).astype('int8')
train_labels = pl.from_pandas(train_labels)
train_labels.head()
train_labels = train_labels.explode('ground_truth').with_columns([pl.col('ground_truth').alias('aid'), pl.lit(1).alias('label')]).with_columns([
    pl.col('ground_truth').cast(pl.Int32),
    pl.col('session').cast(pl.Int32),
    pl.col('aid').cast(pl.Int32),
])

train_df_click_all_sampled =  train_df_click_all.sample(n= int(len(train_df_click_all)*0.5))

train_df_click_all_sampled = train_df_click_all_sampled.join(train_labels, how='left', on=['session', 'type', 'aid']).with_column(pl.col('label').fill_null(0))

In [ ]:
%%time

# Training the model. Seems LGBMRanker train pretty fast. Should be able to add more features
from lightgbm.sklearn import LGBMRanker

ranker = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    boosting_type="dart",
    n_estimators=20,
    importance_type='gain',
)

feature_cols = [
 'ts',
 'real_action',
 'CG_ranking',
 'orders',
 'clicks',
 'carts',
 'carts_user_counter',
 'clicks_user_counter',
 'orders_user_counter',
 'carts_timed_global_counter',
 'orders_timed_global_counter',
 'clicks_timed_global_counter',]

target = 'label'

def get_session_lenghts(df):
    return df.groupby('session').agg([
        pl.col('session').count().alias('session_length')
    ])['session_length'].to_numpy()

session_lengths_train = get_session_lenghts(train_df_click_all_sampled)

ranker = ranker.fit(
    train_df_click_all_sampled[feature_cols].to_pandas(),
    train_df_click_all_sampled[target].to_pandas(),
    group=session_lengths_train,
)


# Step 2: Model Inference

## Step 2.1: Loading testing data and pre-calculated co-visitation Matrix
For quicker experiment, we will only use 1/10 of the validation test_parquet 

We use another set of session to mimic the test set pattern. 

We extract a different set of sesion using the function  test_df[test_df['session']%10 == 0]

In [ ]:
def load_test():    
    dfs = []
    for e, chunk_file in enumerate(glob.glob('../input/otto-validation/test_parquet/*')):
        chunk = pd.read_parquet(chunk_file)
        chunk.ts = (chunk.ts/1000).astype('int32')
        chunk['type'] = chunk['type'].map(type_labels).astype('int8')
        dfs.append(chunk)
    return pd.concat(dfs).reset_index(drop=True) #.astype({"ts": "datetime64[ms]"})

test_df = load_test()
test_df = test_df[test_df['session']%10 == 0]
print('Test data has shape',test_df.shape)
test_df.head()

We have pre-calcaulted the covisitatioin matrix and saved them as the parquet file. Here we only load the result matrix from kaggle dataset to save time<br>
To understand how to generate the co-vistation matrix, you can see here:
https://www.kaggle.com/code/cdeotte/candidate-rerank-model-lb-0-575

In [ ]:
%%time
# Improved speed for 2X using polars. 
def pqt_to_dict(path):
    return pl.read_parquet(path).groupby('aid_x').agg(pl.col('aid_y').list()).to_pandas().set_index('aid_x').aid_y.apply(list).to_dict()

DISK_PIECES = 4

# LOAD THREE CO-VISITATION MATRICES
top_20_clicks = pqt_to_dict(f'/kaggle/input/otto-covisitation-matrix-parquet-files/top_20_clicks_v{VER}_0.pqt')

for k in range(1,DISK_PIECES): 
    top_20_clicks.update(pd.read_parquet(f'/kaggle/input/otto-covisitation-matrix-parquet-files/top_20_clicks_v{VER}_{k}.pqt') ) 


top_20_buys = pqt_to_dict(f'/kaggle/input/otto-covisitation-matrix-parquet-files/top_15_carts_orders_v{VER}_0.pqt') 

for k in range(1,DISK_PIECES): 
    top_20_buys.update( pqt_to_dict( f'/kaggle/input/otto-covisitation-matrix-parquet-files/top_15_carts_orders_v{VER}_{k}.pqt') )

top_20_buy2buy = pqt_to_dict(f'/kaggle/input/otto-covisitation-matrix-parquet-files/top_15_buy2buy_v{VER}_0.pqt') 

print('Here are size of our 3 co-visitation matrices:')
print( len( top_20_clicks ), len( top_20_buy2buy ), len( top_20_buys ) )

# Step 2.2 Candidate Generation using ReRank Model

Same logic with the candidate ReRank model. There is a changes made for the clicks suggestion (See the comment below)

In [ ]:
top_clicks = test_df.loc[test_df['type']== 0,'aid'].value_counts().index.values[:20] 
top_carts = test_df.loc[test_df['type']== 1,'aid'].value_counts().index.values[:20]
top_orders = test_df.loc[test_df['type']== 2,'aid'].value_counts().index.values[:20]

In [ ]:
def suggest_clicks(df):
    # USER HISTORY AIDS AND TYPES
    aids=df.aid.tolist()
    types = df.type.tolist()
    unique_aids = list(dict.fromkeys(aids[::-1] ))
    # RERANK CANDIDATES USING WEIGHTS
    if len(unique_aids)>=20:
        weights=np.logspace(0.1,1,len(aids),base=2, endpoint=True)-1
        aids_temp = Counter() 
        # RERANK BASED ON REPEAT ITEMS AND TYPE OF ITEMS
        for aid,w,t in zip(aids,weights,types): 
            aids_temp[aid] += w * type_weight_multipliers[t]
        sorted_aids = [k for k,v in aids_temp.most_common(20)]
        return sorted_aids
    # USE "CLICKS" CO-VISITATION MATRIX
    aids2 = list(itertools.chain(*[top_20_clicks[aid] for aid in unique_aids if aid in top_20_clicks]))
    # RERANK CANDIDATES
    top_aids2 = [aid2 for aid2, cnt in Counter(aids2).most_common(20) if aid2 not in unique_aids]    
    result = unique_aids + top_aids2[:20 - len(unique_aids)]
    # USE TOP20 TEST CLICKS
    return result + list(top_clicks)[:20-len(result)]

I have re-write the function and made the changes below

1. Changing recommended aid number from 20 to 40
The reason is that if only 20 is recommended, the ranker actually will not boost perofrmance since the LeaderBoard Recall score is calcualted **regardless** of the order. <br>
Therefore, here we recommnd more candidate to increase the **Recall metrics (i.e. Total coverage of ground truth aid that is recommeneded in the candidate recommendation)**

2. aid in test set will not be recommened
The aid in test set will be handled seperately since they have ts information and should not have CG_ranking (Candidate generation ranking) feature info.

In [ ]:
top_clicks = test_df.loc[test_df['type']== 0,'aid'].value_counts().index.values[:40] 
def suggest_clicks_40_candidates(df):
    # USER HISTORY AIDS AND TYPES
    aids=df.aid.tolist()
    types = df.type.tolist()
    unique_aids = list(dict.fromkeys(aids[::-1] ))
    # RERANK CANDIDATES USING WEIGHTS
    if len(unique_aids)>=40:
        weights=np.logspace(0.1,1,len(aids),base=2, endpoint=True)-1
        aids_temp = Counter() 
        # RERANK BASED ON REPEAT ITEMS AND TYPE OF ITEMS
        for aid,w,t in zip(aids,weights,types): 
            aids_temp[aid] += w * type_weight_multipliers[t]
        sorted_aids = [k for k,v in aids_temp.most_common(40)]
        return sorted_aids
    # USE "CLICKS" CO-VISITATION MATRIX
    aids2 = list(itertools.chain(*[top_20_clicks[aid] for aid in unique_aids if aid in top_20_clicks]))
    # RERANK CANDIDATES
    top_aids2 = [aid2 for aid2, cnt in Counter(aids2).most_common(40) if aid2 not in unique_aids]    
    result = top_aids2[:40]
    # USE TOP20 TEST CLICKS
    return result + list(top_clicks)[:40-len(result)]

In [ ]:
def suggest_carts(df):
    # User history aids and types
    aids = df.aid.tolist()
    types = df.type.tolist()
    
    # UNIQUE AIDS AND UNIQUE BUYS
    unique_aids = list(dict.fromkeys(aids[::-1] ))
    df = df.loc[(df['type'] == 0)|(df['type'] == 1)]
    unique_buys = list(dict.fromkeys(df.aid.tolist()[::-1]))
    
    # Rerank candidates using weights
    if len(unique_aids) >= 20:
        weights=np.logspace(0.5,1,len(aids),base=2, endpoint=True)-1
        aids_temp = Counter() 
        
        # Rerank based on repeat items and types of items
        for aid,w,t in zip(aids,weights,types): 
            aids_temp[aid] += w * type_weight_multipliers[t]
        
        # Rerank candidates using"top_20_carts" co-visitation matrix
        aids2 = list(itertools.chain(*[top_20_buys[aid] for aid in unique_buys if aid in top_20_buys]))
        for aid in aids2: aids_temp[aid] += 0.1
        sorted_aids = [k for k,v in aids_temp.most_common(20)]
        return sorted_aids
    
    # Use "cart order" and "clicks" co-visitation matrices
    aids1 = list(itertools.chain(*[top_20_clicks[aid] for aid in unique_aids if aid in top_20_clicks]))
    aids2 = list(itertools.chain(*[top_20_buys[aid]*2 for aid in unique_aids if aid in top_20_buys]))
    
    # RERANK CANDIDATES
    top_aids2 = [aid2 for aid2, cnt in Counter(aids1+aids2).most_common(20) if aid2 not in unique_aids] 
    result = unique_aids + top_aids2[:20 - len(unique_aids)]
    
    # USE TOP20 TEST ORDERS

    return result + list(top_carts)[:20-len(result)]

In [ ]:
def suggest_buys(df):
    
    # USER HISTORY AIDS AND TYPES
    aids=df.aid.tolist()
    types = df.type.tolist()
    # UNIQUE AIDS AND UNIQUE BUYS
    unique_aids = list(dict.fromkeys(aids[::-1] ))
    df = df.loc[(df['type']==1)|(df['type']==2)]
    unique_buys = list(dict.fromkeys( df.aid.tolist()[::-1] ))
    # RERANK CANDIDATES USING WEIGHTS
    if len(unique_aids)>=20:
        weights=np.logspace(0.5,1,len(aids),base=2, endpoint=True)-1
        aids_temp = Counter() 
        # RERANK BASED ON REPEAT ITEMS AND TYPE OF ITEMS
        for aid,w,t in zip(aids,weights,types): 
            aids_temp[aid] += w * type_weight_multipliers[t]
        # RERANK CANDIDATES USING "BUY2BUY" CO-VISITATION MATRIX
        aids3 = list(itertools.chain(*[top_20_buy2buy[aid] for aid in unique_buys if aid in top_20_buy2buy]))
        for aid in aids3: aids_temp[aid] += 0.1
        sorted_aids = [k for k,v in aids_temp.most_common(20)]
        return sorted_aids
    # USE "CART ORDER" CO-VISITATION MATRIX
    aids2 = list(itertools.chain(*[top_20_buys[aid] for aid in unique_aids if aid in top_20_buys]))
    # USE "BUY2BUY" CO-VISITATION MATRIX
    aids3 = list(itertools.chain(*[top_20_buy2buy[aid] for aid in unique_buys if aid in top_20_buy2buy]))
    # RERANK CANDIDATES
    top_aids2 = [aid2 for aid2, cnt in Counter(aids2+aids3).most_common(20) if aid2 not in unique_aids] 
    result = unique_aids + top_aids2[:20 - len(unique_aids)]
    # USE TOP20 TEST ORDERS
    return result + list(top_orders)[:20-len(result)]

### Generate candidates for each action using co-visitation matrix

In [ ]:
from pandarallel import pandarallel

In [ ]:
# Using pandarallel to accelerate
from pandarallel import pandarallel
pandarallel.initialize(nb_workers = 4,progress_bar=True)

In [ ]:
%%time
# Improved speed for 2X using pandarallel
# pred_df_clicks = test_df.sort_values(["session", "ts"]).groupby(["session"]).parallel_apply(
#     lambda x: suggest_clicks(x)
# )

# Improved speed for 2X using pandarallel
pred_df_clicks = test_df.sort_values(["session", "ts"]).groupby(["session"]).parallel_apply(
    lambda x: suggest_clicks_40_candidates(x)
)

pred_df_buys = test_df.sort_values(["session", "ts"]).groupby(["session"]).parallel_apply(
    lambda x: suggest_buys(x)
)

pred_df_carts = test_df.sort_values(["session", "ts"]).groupby(["session"]).parallel_apply(
    lambda x: suggest_carts(x)
)

# Step 2.3 Candidate Ranking using LGBM Ranker

To run the ranker, we need to combine two sources of data
1. aid provided by the test set
2. aid recommended in the candidate generation stage

These two types of aid should be handled seperately since some of their feature is different (e.g. aid from candidate generation does not haev ts information)

### Handle aid from test set

In [ ]:
# Extracting the testing data action for clicks only
test_df_click = test_df[test_df['type'] == 0]
test_df_click['real_action'] = 1
test_df_click['CG_ranking'] = 0

# Calculate the last three aid for the embedding calculation
test_df_click['aid_last'] = test_df_click.groupby(['session']).aid.shift(1).bfill()
test_df_click['aid_second_last'] = test_df_click.groupby(['session']).aid.shift(2).bfill()
test_df_click['aid_third_last'] = test_df_click.groupby(['session']).aid.shift(3).bfill()

test_df_click = pl.from_pandas(test_df_click)

test_df_click = test_df_click.with_columns([
    pl.col('aid_last').cast(pl.Int32),
    pl.col('aid_second_last').cast(pl.Int32),
    pl.col('aid_third_last').cast(pl.Int32),
])

### Handle aid from candidate recommenedation

In [ ]:
# Extracting the candiddate generated from the testing data
clicks_candidate_df = pl.from_pandas(pd.DataFrame(pred_df_clicks, columns = ['aid']).reset_index())
clicks_candidate_df = clicks_candidate_df.explode('aid')
clicks_candidate_df = clicks_candidate_df.with_columns([
    pl.lit(0).alias('ts').cast(pl.Int32),    
    pl.lit(0).alias('type').cast(pl.Int8),
    pl.lit(0).alias('real_action').cast(pl.Int64),
    pl.col('session').cast(pl.Int32),
    pl.col('aid').cast(pl.Int32),
])
n_col_after_join = clicks_candidate_df.groupby('session').agg([
    pl.col('aid').cumcount().alias('CG_ranking')]).select(
    pl.col('CG_ranking').explode().cast(pl.Int64))

test_df_click_recommended = pl.concat([clicks_candidate_df, n_col_after_join], how="horizontal")


### Feature Calculation for inference data

In [ ]:
%%time

# Calculating the embedding of last three actions aid
test_df_click = test_df_click.join(embedding_weigh_dict_df, on = 'aid', how = 'left').join(embedding_weigh_dict_df, right_on = 'aid', left_on = 'aid_last', how = 'left', suffix = 'last_1').join(embedding_weigh_dict_df, right_on = 'aid', left_on = 'aid_second_last', how = 'left', suffix = 'last_2').join(embedding_weigh_dict_df, right_on = 'aid', left_on = 'aid_third_last', suffix = 'last_3').select(pl.exclude(['aid_last', 'aid_second_last', 'aid_third_last']))

In [ ]:
# Calculating the embedding of last three actions aid for the recommended action. We all use the last three actual aid embedding for the last three aid
train_df_click_last_action_embedding = test_df_click.sort(['session', 'ts']).groupby(['session']).last().select(pl.exclude(['aid', 'ts','type','real_action', 'CG_ranking', 'aid_last', 'aid_second_last', 'aid_third_last'] + ['Embedding_' + str(i) for i in range(32)]))

test_df_click_recommended = test_df_click_recommended.join(embedding_weigh_dict_df, on = 'aid', how = 'left').join(train_df_click_last_action_embedding, on = 'session', how = 'left')

In [ ]:

# Calculating the feature on the inference data
test_df_click = test_df_click.join(aid_global_counter_all_types, on='aid', suffix ='_global_counter').join(aid_global_user_counter_all_types, on='aid', suffix ='_user_counter').join(aid_global_user_counter_all_types_time_weighted, on='aid', suffix ='_timed_global_counter')
test_df_click_recommended = test_df_click_recommended.join(aid_global_counter_all_types, on='aid', suffix ='_global_counter').join(aid_global_user_counter_all_types, on='aid', suffix ='_user_counter').join(aid_global_user_counter_all_types_time_weighted, on='aid', suffix ='_timed_global_counter')

# Combining the actual test data aid and recommended aid
test_df_click_all = pl.concat([test_df_click, test_df_click_recommended], how = 'vertical')

In [ ]:
# Model inference
scores = ranker.predict(test_df_click_all[feature_cols].to_pandas())

# Appending the model score to the original dataframe
test_df_click_all = test_df_click_all.with_columns(pl.Series(name='score', values=scores))

# Getting the top 20 candidates from the prediction
clicks_pred_df = test_df_click_all.sort(['session', 'score'], reverse=True).groupby('session').agg([
    pl.col('aid').limit(20).list().alias('labels')
])

# Converting to pandas format and making it align with result format
clicks_pred_df = clicks_pred_df.with_columns(
pl.col('session') + '_clicks'
).to_pandas()

# Step 2.4 Exporting to csv

In [ ]:
# clicks_pred_df = pd.DataFrame(pred_df_clicks.add_suffix("_clicks"), columns=["labels"]).reset_index()
orders_pred_df = pd.DataFrame(pred_df_buys.add_suffix("_orders"), columns=["labels"]).reset_index()
carts_pred_df = pd.DataFrame(pred_df_carts.add_suffix("_carts"), columns=["labels"]).reset_index()

In [ ]:
pred_df = pd.concat([clicks_pred_df, orders_pred_df, carts_pred_df])
pred_df.columns = ["session_type", "labels"]
pred_df["labels"] = pred_df.labels.apply(lambda x: " ".join(map(str,x)))
pred_df.to_csv("validation_preds.csv", index=False)
pred_df.head()

# Step 3: Model Evaluation

This code is from Chris. It aims to calculate the local Recall score of the ranking. See the notebook link here: https://www.kaggle.com/code/cdeotte/compute-validation-score-cv-565

In [ ]:
# # FREE MEMORY
# del pred_df_clicks, pred_df_buys, clicks_pred_df, orders_pred_df, carts_pred_df
# del top_20_clicks, top_20_buy2buy, top_20_buys, top_clicks, top_orders, test_df
# _ = gc.collect()

In [ ]:
%%time
# COMPUTE METRIC
score = 0
weights = {'clicks': 0.10, 'carts': 0.30, 'orders': 0.60}
for t in ['clicks','carts','orders']:
    sub = pred_df.loc[pred_df.session_type.str.contains(t)].copy()
    sub['session'] = sub.session_type.apply(lambda x: int(x.split('_')[0]))
    sub.labels = sub.labels.apply(lambda x: [int(i) for i in x.split(' ')])
    test_labels = pd.read_parquet('../input/otto-validation/test_labels.parquet')
    test_labels = test_labels.loc[test_labels['type']==t]
    test_labels = test_labels.merge(sub, how='left', on=['session'])
    test_labels = test_labels.dropna()
    test_labels['hits'] = test_labels.apply(lambda df: len(set(df.ground_truth).intersection(set(df.labels))), axis=1)
    test_labels['gt_count'] = test_labels.ground_truth.str.len().clip(0,20)
    recall = test_labels['hits'].sum() / test_labels['gt_count'].sum()
    score += weights[t]*recall
    print(f'{t} recall =',recall)
    
print('=============')
print('Overall Recall =',score)
print('=============')

### Previous performance of the ReRank Model

clicks recall = 0.3896982547336502 <br>
carts recall = 0.4105610333097339<br>
orders recall = 0.6519093116387601 <br>
=============<br>
Overall Recall = 0.5532837224495413 <br>
=============<br>


**The performance drop for the click behavior, we may need to optimize the model or put more weight to the aid of actual behavior.**